# Fun

Purpose of this notebook is to actually do training to discover different model 
configurations that would work to train a pretty good TinyGPT. 

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


cuda
Tesla T4


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
!git clone https://github.com/sanchitram1/242b-hw3
%cd 242b-hw3

Cloning into '242b-hw3'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 60 (delta 30), reused 47 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 471.86 KiB | 1.86 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/242b-hw3


In [4]:
from tokenizer import build_tokenizer

In [5]:
from config import (
    DataConfig,
    GlobalTrainingConfig,
    RunConfig,
    TokenConfig,
    TokenizationConfig,
)


token_config = TokenConfig()
data_config = DataConfig()
global_training_config = GlobalTrainingConfig()
tokenization_config = TokenizationConfig()
run_config = RunConfig("long-100000")

Created /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/long-100000 and subfolders


In [6]:
tokenizer = build_tokenizer(
    tokenization_config, token_config, data_config.training_file_colab
)


Found it in /content/drive/MyDrive/courses/242B/HW3/artifacts/shared/tokenizers/tinystories_bpe_metaspace_3000_1000000.json, loading from there


In [7]:
from tokenizer import count_tokens, build_token_memmap

In [8]:
train_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.training_file_colab
)
valid_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.validation_file_colab
)
print(f"There are {train_token_count} tokens in train, {valid_token_count} valid")

train_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.training_file_colab,
    train_token_count,
)
valid_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.validation_file_colab,
    valid_token_count,
)

There are 189162345 tokens in train, 5192964 valid


In [9]:
from models import TokenChunkDataset

In [10]:
train_dataset = TokenChunkDataset(
    train_token_memmap_path, train_token_count, global_training_config.context_length
)
valid_dataset = TokenChunkDataset(
    valid_token_memmap_path, valid_token_count, global_training_config.context_length
)

In [12]:
from config import ModelConfig

In [13]:
configs = [
    ModelConfig(
        name="small",
        d_model=128,
        n_heads=4,
        n_layers=3,
        d_ff=384,
        batch_size=16,
        learning_rate=3e-4,
        weight_decay=0.1,
        warmup_steps=50,
        max_steps=100000,
        use_amp=torch.cuda.is_available(),
    ),
    ModelConfig(
        name="base",
        d_model=160,
        n_heads=5,
        n_layers=4,
        d_ff=480,
        batch_size=16,
        learning_rate=3e-4,
        weight_decay=0.1,
        warmup_steps=80,
        max_steps=100000,
        use_amp=torch.cuda.is_available(),
    ),
    ModelConfig(
        name="large",
        d_model=192,
        n_heads=6,
        n_layers=5,
        d_ff=576,
        batch_size=12,
        learning_rate=2.5e-4,
        weight_decay=0.1,
        warmup_steps=80,
        max_steps=100000,
        use_amp=torch.cuda.is_available(),
    ),
]


In [14]:
from training import train_model
import json
from utils import save_json
from plot import plot_training_curves, plot_validation_curves

In [15]:
global_training_config.context_window = global_training_config.context_length

In [16]:
results: dict[str, dict] = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for config in configs:
    metrics_path = run_config.metrics / f"{config.name}.json"
    if metrics_path.exists():
        results[config.name] = json.loads(metrics_path.read_text(encoding="utf-8"))
        continue
    result = train_model(
        run_config,
        global_training_config,
        config,
        tokenizer,
        train_dataset,
        valid_dataset,
        device=device,
    )
    results[config.name] = result
    save_json(result, metrics_path)

    if device.type == "cuda":
        torch.cuda.empty_cache()

plot_training_curves(results, run_config.plots / "training_loss_all_models.png")
plot_validation_curves(results, run_config.plots / "validation_loss_all_models.png")


[small] Loss(step=1)=8.04
[small] Loss(step=1000)=6.49
[small] Loss(step=2000)=5.50
[small] Loss(step=3000)=4.96
[small] Loss(step=4000)=5.00
[small] Loss(step=5000)=4.69
[small] Loss(step=6000)=4.49
[small] Loss(step=7000)=4.17
[small] Loss(step=8000)=4.27
[small] Loss(step=9000)=4.16
[small] Loss(step=10000)=4.08
[small] Loss(step=11000)=3.88
[small] Loss(step=12000)=3.89
[small] Loss(step=13000)=3.75
[small] Loss(step=14000)=3.84
[small] Loss(step=15000)=3.61
[small] Loss(step=16000)=3.70
[small] Loss(step=17000)=3.55
[small] Loss(step=18000)=3.78
[small] Loss(step=19000)=3.49
[small] Loss(step=20000)=3.69
[small] Loss(step=21000)=3.57
[small] Loss(step=22000)=3.35
[small] Loss(step=23000)=3.31
[small] Loss(step=24000)=3.59
[small] Loss(step=25000)=3.29
[small] Loss(step=26000)=3.36
[small] Loss(step=27000)=3.34
[small] Loss(step=28000)=3.25
[small] Loss(step=29000)=3.32
[small] Loss(step=30000)=3.34
[small] Loss(step=31000)=3.26
[small] Loss(step=32000)=3.50
[small] Loss(step=33000

In [38]:
from models import TinyGPT
from tokenizers import Tokenizer
import torch.nn.functional as F

SAMPLE_PROMPTS = (
    "Once upon a time",
    "A little girl and her pet cat",
    "A fairy woke up",
    "A lion, a bear, and a penguin",
    "The two children",
)


def model_checkpoint_path(run_config: RunConfig, model_config: ModelConfig):
    return run_config.models / f"{model_config.name}.pt"


def load_model(
    model_path,
    device: torch.device,
) -> TinyGPT:
    """Loads a saved TinyGPT checkpoint and returns it in eval mode."""
    checkpoint = torch.load(model_path, map_location=device)
    config = ModelConfig(**checkpoint["config"])
    model = TinyGPT(
        vocab_size=checkpoint["vocab_size"],
        context_length=checkpoint["context_length"],
        d_model=config.d_model,
        n_heads=config.n_heads,
        n_layers=config.n_layers,
        d_ff=config.d_ff,
        dropout=config.dropout,
    ).to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model


def top_k_filter(logits: torch.Tensor, top_k: int) -> torch.Tensor:
    if top_k <= 0 or top_k >= logits.numel():
        return logits
    values, _ = torch.topk(logits, top_k)
    cutoff = values[..., -1, None]
    return torch.where(logits < cutoff, torch.full_like(logits, float("-inf")), logits)


@torch.no_grad()
def generate_text(
    model: TinyGPT,
    tokenizer: Tokenizer,
    prompt: str,
    device: torch.device,
    max_new_tokens: int = 120,
    temperature: float = 0.9,
    top_k: int = 40,
) -> str:
    ids = tokenizer.encode(prompt).ids
    if not ids:
        ids = [tokenizer.token_to_id(token_config.bos)]

    eos_id = tokenizer.token_to_id(token_config.eos)
    tokens = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        input_tokens = tokens[:, -global_training_config.context_length :]
        logits = model(input_tokens)[:, -1, :] / max(temperature, 1e-5)
        logits = top_k_filter(logits, top_k=top_k)
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_token], dim=1)
        if eos_id is not None and int(next_token.item()) == eos_id:
            break
    decoded = tokenizer.decode(tokens[0].tolist(), skip_special_tokens=True)
    return decoded

In [23]:
model_path = model_checkpoint_path(run_config, configs[-1])
print(model_path)
model = load_model(model_path, device)

/content/drive/MyDrive/courses/242B/HW3/artifacts/runs/long-100000/models/large.pt


In [26]:
from IPython.display import Markdown, Image

In [39]:
generations: list[dict] = []
output_settings = [
    {"temperature": 0.7, "top_k": 30},  # balanced
    {"temperature": 0.5, "top_k": 10},  # conservative
    {"temperature": 0.8, "top_k": 50},  # experimental
]

for prompt in SAMPLE_PROMPTS:
    for config in configs:
        model_path = model_checkpoint_path(run_config, config)
        model = load_model(model_path, device)

        for setting in output_settings:
            temperature = setting["temperature"]
            top_k = setting["top_k"]
            generation = generate_text(
                model, tokenizer, prompt, device, 120, temperature, top_k
            )

            generations.append(
                {
                    "model": config.name,
                    "prompt": prompt,
                    "generated_text": generation,
                    "temperature": temperature,
                    "top_k": top_k,
                }
            )

In [35]:
for sample in generations:
    name = sample["model"]
    display(
        Markdown(
            f"**{name}**: Prompt=`{sample['prompt']}` *({sample['temperature']}, {sample['top_k']})*"
        )
    )
    display(Markdown(sample["generated_text"]))


**small**: Prompt=`Once upon a time` *(0.7, 30)*

Once upon a time there was a big tree. She loved to climb the tree and make pretty leaves. One day, the tree was flying high in the sky. It was a big tree. The tree was very hot in the tree. The tree was sad and hungry. The tree was not scared anymore. It was a good tree!

**small**: Prompt=`Once upon a time` *(0.5, 10)*

Once upon a time there was a little girl named Lily. Lily had a big toy box that she loved to play with her ball. One day, she went to the park with her mom. She saw a big box of toys. Lily wanted to play with the box for her toe. She took the box and put it back to her room. Lily was very happy. She said, "Thank you, Lily!" Lily was happy and played with her toys all day.

**small**: Prompt=`Once upon a time` *(0.8, 50)*

...

A little girl and her pet cat named Lily, went for a walk. They saw a big tree with many leaves. Max was very happy to have a friend to play. Lily said, "Let's see that a little bird in the tree!" So, they started to play catch with the bird in the sun. They were very happy and full. The bird knew that Max was right. Max and Lily played with the bird. They were very good friends and had lots of fun.

**large**: Prompt=`A little girl and her pet cat` *(0.5, 10)*

A little girl and her pet cat were very happy. They played together all day long, and the little girl was not scared anymore.

**large**: Prompt=`A little girl and her pet cat` *(0.8, 50)*

A little girl and her pet cat were shocked. They looked at each other and decided to get her ball back. Amy said: "Let's get the ball down and watch the ball go away. Maybe it is too big and can get it out."
They tried to grab the ball, but it was too hard for Amy and her best. They were too afraid. Then, Amy started to cry and ran around the house. She saw the ball near the door and knew how much it was. She tried to get it, but it was too late. The ball was too big for her to get out. Amy was sad and scared. She tried to help, but the ball did not move.

In [40]:
samples_dir = run_config.run_dir / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)

generations_path = samples_dir / "generations.json"
generations_path.write_text(
    json.dumps(generations, indent=2),
    encoding="utf-8",
)

print(f"Saved generations to {generations_path}")


Saved generations to /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/long-100000/samples/generations.json


In [37]:
from dataclasses import asdict

manifest = {
    "run_id": run_config.run_id,
    "run_dir": str(run_config.run_dir),
    "device": str(device),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "data": {
        "training_file": str(data_config.training_file),
        "validation_file": str(data_config.validation_file),
    },
    "tokenization": asdict(tokenization_config),
    "global_training": asdict(global_training_config),
    "models": [asdict(config) for config in configs],
    "artifacts": {
        "models_dir": str(run_config.models),
        "metrics_dir": str(run_config.metrics),
        "plots_dir": str(run_config.plots),
        "samples_dir": str(samples_dir),
        "generations": str(generations_path),
    },
    "generation_settings": output_settings,
}

manifest_path = run_config.run_dir / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(f"Saved manifest to {manifest_path}")


Saved manifest to /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/long-100000/manifest.json
